In [1]:
import os, sys, time, json, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A

REPO_DIR = r'C:\Users\Debabrata\Desktop\Bright\BRIGHT\SEG_REASONING-main'
DATA_DIR = r'C:\Users\Debabrata\Desktop\Bright\BRIGHT\acdc_data\ACDC'

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'models'))

print('torch:', torch.__version__)
print('CUDA :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU  :', torch.cuda.get_device_name(0))
    print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')


torch: 2.11.0+cu126
CUDA : True
GPU  : NVIDIA RTX A5500
VRAM : 24.1 GB


In [2]:
import gdown
from pathlib import Path

ACDC_DIR = r'C:\Users\Debabrata\Desktop\Bright\BRIGHT\acdc_data'
Path(ACDC_DIR).mkdir(parents=True, exist_ok=True)

# Download from Google Drive
FILE_ID  = '1CruCQ-jjvA97BX-LIYwXaRMLmp3DN9zc'
OUT_PATH = str(Path(ACDC_DIR) / 'acdc.zip')

print('Downloading ACDC dataset...')
gdown.download(f'https://drive.google.com/uc?id={FILE_ID}', OUT_PATH, quiet=False)

# Unzip
import zipfile
print('Extracting...')
with zipfile.ZipFile(OUT_PATH, 'r') as z:
    z.extractall(ACDC_DIR)

# Check what we got
import os
for root, dirs, files in os.walk(ACDC_DIR):
    level = root.replace(ACDC_DIR, '').count(os.sep)
    if level < 3:
        print('  ' * level + os.path.basename(root) + '/')
        if level == 2:
            print('  ' * (level+1) + f'{len(files)} files')

print('Done! Update DATA_DIR in Cell 2 based on the structure above.')

Downloading...
From (original): https://drive.google.com/uc?id=1CruCQ-jjvA97BX-LIYwXaRMLmp3DN9zc
From (redirected): https://drive.google.com/uc?id=1CruCQ-jjvA97BX-LIYwXaRMLmp3DN9zc&confirm=t&uuid=150d717b-a294-4b35-885a-d4ecabe43efa
To: C:\Users\Debabrata\Desktop\Bright\BRIGHT\acdc_data\acdc.zip
100%|██████████| 111M/111M [00:11<00:00, 9.45MB/s] 


Extracting...
acdc_data/
  ACDC/
    lists_ACDC/
      3 files
    test/
      40 files
    train/
      1304 files
    valid/
      182 files
  __MACOSX/
    ACDC/
      4 files
Done! Update DATA_DIR in Cell 2 based on the structure above.


In [3]:
NUM_CLASSES   = 4       # BG, RV, Myo, LV
IMAGE_SIZE    = 224     # native ACDC resolution
BATCH_SIZE    = 16
EPOCHS        = 200
LR_ENCODER    = 1e-4
LR_DECODER    = 1e-3
LATENT_H      = 640     # same as run_005
LATENT_Z      = 384     # same as run_005
SEED          = 42
VAL_FRACTION  = 0.1     # carve 10% from train
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

print('Device:', DEVICE)
print(f'Classes: {NUM_CLASSES} — BG=0, RV=1, Myo=2, LV=3')
print(f'Latents: h={LATENT_H}, z={LATENT_Z}')


Device: cuda
Classes: 4 — BG=0, RV=1, Myo=2, LV=3
Latents: h=640, z=384


In [20]:
class ACDCDataset(Dataset):
    def __init__(self, files, augment=False):
        self.files   = sorted(files)
        self.augment = augment
        if augment:
            self.aug = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.3),
                A.Rotate(limit=30, p=0.5),
                A.RandomBrightnessContrast(p=0.3),
            ])

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        d     = np.load(self.files[idx])
        img   = d['img'].astype(np.float32)
        label = d['label'].astype(np.int64)

    # Normalize [-1,1] -> [0,1]
        img = (img + 1.0) / 2.0

    # Handle 3D volumes — take middle slice for training,
    # but for test volumes we return all slices stacked
        if img.ndim == 3:
            mid   = img.shape[0] // 2
            img   = img[mid]
            label = label[mid]

        if self.augment:
            out   = self.aug(image=img, mask=label.astype(np.uint8))
            img   = out['image']
            label = out['mask'].astype(np.int64)

    # Grayscale -> 3 channel
        img_3ch = np.stack([img, img, img], axis=0)
        return {
        'image': torch.from_numpy(img_3ch).float(),
        'label': torch.from_numpy(label).long(),
    }


def build_dataloaders():
    train_files = list(Path(DATA_DIR, 'train').glob('*.npz'))
    test_files  = list(Path(DATA_DIR, 'test').glob('*.npz'))

    # Patient-level val split from train cases (not random slice split)
    train_cases = list(set(f.name.split('_slice')[0] for f in train_files))
    random.shuffle(train_cases)
    n_val_cases  = 10  # standard: 10 val cases
    val_cases    = set(train_cases[:n_val_cases])
    train_cases  = set(train_cases[n_val_cases:])

    val_files   = [f for f in train_files if f.name.split('_slice')[0] in val_cases]
    train_files = [f for f in train_files if f.name.split('_slice')[0] in train_cases]

    print(f'Train={len(train_files)} slices ({len(train_cases)} cases)')
    print(f'Val  ={len(val_files)} slices ({n_val_cases} cases)')
    print(f'Test ={len(test_files)} volumes (20 cases × 2 phases)')

    return (
        DataLoader(ACDCDataset(train_files, augment=True),
                   batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True),
        DataLoader(ACDCDataset(val_files,   augment=False),
                   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
        DataLoader(ACDCDataset(test_files,  augment=False),
                   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    )


train_loader, val_loader, test_loader = build_dataloaders()


Train=1124 slices (60 cases)
Val  =180 slices (10 cases)
Test =40 volumes (20 cases × 2 phases)


In [21]:
# Run this in a new cell to inspect the encoder
from v7_model import ConvNeXTEncoder
enc = ConvNeXTEncoder('convnext_tiny', pretrained=False)
print(dir(enc))
print()
# Check what attributes exist
for attr in ['channels', 'out_channels', 'channel_list', 'ch']:
    print(attr, ':', hasattr(enc, attr))

['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_is_full_backward_hook', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full_backward_hook', '_modules', '_named_members', '_non_persistent_buffers_set', '_parameters', '_register_load_state_dict_pre_hoo

In [22]:
import torch
enc = ConvNeXTEncoder('convnext_tiny', pretrained=False)
x = torch.rand(1, 3, 256, 256)
with torch.no_grad():
    out = enc(x)
print(type(out))
if isinstance(out, (list, tuple)):
    for i, o in enumerate(out):
        print(f'  [{i}]: {tuple(o.shape)}')
else:
    print(tuple(out.shape))

<class 'tuple'>
  [0]: (1, 96, 64, 64)
  [1]: (1, 192, 32, 32)
  [2]: (1, 384, 16, 16)
  [3]: (1, 768, 8, 8)


In [23]:
for m in list(sys.modules):
    if 'v7_train' in m or m.endswith('_model'):
        del sys.modules[m]

from v7_model import (
    ConvNeXTEncoder, LowLevelRefinement,
    HighLevelTransition, UpFuse, ResBlock,
    UncertaintyHead, HaltHead, _safe_groups,
)

# ConvNeXt-tiny channel dims (verified from forward pass)
CH = {'s1': 96, 's2': 192, 's3': 384, 's4': 768}


class MultiClassDecoder(nn.Module):
    def __init__(self, dz, s1_ch, s2_ch, s3_ch, num_classes=4):
        super().__init__()
        out3 = max(s3_ch, dz // 2)
        out2 = max(s2_ch, dz // 4)
        out1 = max(s1_ch, 32)
        self.fuse3  = UpFuse(dz,   s3_ch, out3)
        self.fuse2  = UpFuse(out3, s2_ch, out2)
        self.fuse1  = UpFuse(out2, s1_ch, out1)
        self.refine = ResBlock(out1, out1)
        self.out    = nn.Conv2d(out1, num_classes, 1)

    def forward(self, z, s1, s2, s3, target_size):
        x = self.fuse3(z, s3)
        x = self.fuse2(x, s2)
        x = self.fuse1(x, s1)
        x = self.refine(x)
        x = self.out(x)
        return F.interpolate(x, size=target_size, mode='bilinear', align_corners=False)


class MultiClassHiReMed(nn.Module):
    def __init__(self, latent_dim_h=640, latent_dim_z=384, num_classes=4,
                 max_steps=6, min_steps=3, K_inner=3,
                 halt_improvement_threshold=0.1,
                 dropout_p=0.1, recursive_dropout_p=0.2):
        super().__init__()
        self.encoder   = ConvNeXTEncoder('convnext_tiny', pretrained=True)
        ch = CH  # use hardcoded channel dims
        self.z_proj    = nn.Conv2d(ch['s4'], latent_dim_z, 1, bias=False)
        self.low       = LowLevelRefinement(latent_dim_z, latent_dim_h, ch['s2'],
                                            dropout_p=dropout_p)
        self.high      = HighLevelTransition(latent_dim_h, latent_dim_z, ch['s3'],
                                             dropout_p=dropout_p)
        self.decoder   = MultiClassDecoder(latent_dim_z, ch['s1'], ch['s2'],
                                           ch['s3'], num_classes)
        self.unc_head  = UncertaintyHead(latent_dim_z, ch['s1'])
        self.halt_head = HaltHead(latent_dim_z, latent_dim_h)

        self.num_classes = num_classes
        self.latent_dim_h = latent_dim_h
        self.latent_dim_z = latent_dim_z
        self.max_steps = max_steps
        self.min_steps = min_steps
        self.K_inner   = K_inner
        self.halt_improvement_threshold = halt_improvement_threshold
        self.recursive_dropout_p = recursive_dropout_p

    def forward(self, x, force_steps=None):
        B, _, H, W = x.shape
        dtype = x.dtype; dev = x.device

        # Encoder returns tuple — unpack explicitly
        enc_out = self.encoder(x)
        s1, s2, s3, s4 = enc_out[0], enc_out[1], enc_out[2], enc_out[3]

        z      = self.z_proj(s4)
        h      = torch.zeros(B, self.latent_dim_h, device=dev, dtype=dtype)
        mu     = torch.zeros(B, self.latent_dim_h, device=dev, dtype=dtype)
        logvar = torch.zeros(B, self.latent_dim_h, device=dev, dtype=dtype)

        y_logit     = torch.zeros(B, self.num_classes, H, W, device=dev, dtype=dtype)
        y_prob_prev = torch.zeros(B, 1, H, W, device=dev, dtype=dtype)
        u_prev      = torch.zeros(B, 1, H, W, device=dev, dtype=dtype)

        max_s       = force_steps if force_steps is not None else self.max_steps
        use_halting = force_steps is None
        active      = torch.ones(B, dtype=torch.bool, device=dev)
        outputs     = []

        for step in range(max_s):
            if self.training and self.recursive_dropout_p > 0 and step > 0:
                mask = torch.bernoulli(
                    torch.full((B,1,1,1), 1-self.recursive_dropout_p, device=dev))
                z = z * mask

            for _ in range(self.K_inner):
                z = self.low(z, h, s2, y_prob_prev, u_prev)

            h, mu, logvar = self.high(h, z, s3)
            delta    = self.decoder(z, s1, s2, s3, (H, W))
            y_logit  = y_logit + delta
            y_soft   = F.softmax(y_logit.float(), dim=1)
            u_map    = self.unc_head(z, s1, (H, W)).clamp(0, 1)
            y_class  = (y_soft.argmax(dim=1, keepdim=True).float()
                        / (self.num_classes - 1)).to(dtype)
            y_prob_prev = y_class.detach()
            u_prev      = u_map.detach()
            halt_score  = self.halt_head(z, u_map, h).clamp(0, 1)

            outputs.append({
                'y_logit':     y_logit,
                'y_prob':      y_soft.max(dim=1, keepdim=True)[0].clamp(0,1).to(dtype),
                'uncertainty': u_map,
                'halt_score':  halt_score,
                'mu':          mu,
                'logvar':      logvar,
            })

            if use_halting and step + 1 >= self.min_steps:
                active = active & (halt_score < (1.0 - self.halt_improvement_threshold))
                if not active.any(): break

        return outputs


model = MultiClassHiReMed(
    latent_dim_h=LATENT_H, latent_dim_z=LATENT_Z,
    num_classes=NUM_CLASSES,
).to(DEVICE)

total_M = sum(p.numel() for p in model.parameters()) / 1e6
enc_M   = sum(p.numel() for p in model.encoder.parameters()) / 1e6
print(f'Total  : {total_M:.2f}M')
print(f'Encoder: {enc_M:.2f}M  ({100*enc_M/total_M:.1f}%)')
print(f'Reason : {total_M-enc_M:.2f}M  ({100*(total_M-enc_M)/total_M:.1f}%)')

# Sanity check
model.eval()
with torch.no_grad():
    dummy = torch.rand(2, 3, 224, 224).to(DEVICE)
    out   = model(dummy, force_steps=1)
print(f'Forward OK: y_logit={tuple(out[0]["y_logit"].shape)}')

Total  : 43.22M
Encoder: 27.82M  (64.4%)
Reason : 15.40M  (35.6%)
Forward OK: y_logit=(2, 4, 224, 224)


In [12]:
def multiclass_dice_loss(logits, target, num_classes=4, smooth=1e-5):
    pred    = F.softmax(logits.float(), dim=1)
    target_oh = F.one_hot(target, num_classes).permute(0,3,1,2).float()
    losses  = []
    for c in range(1, num_classes):  # skip BG
        p     = pred[:, c]
        g     = target_oh[:, c]
        inter = (p * g).sum(dim=(1,2))
        union = p.sum(dim=(1,2)) + g.sum(dim=(1,2))
        losses.append(1 - ((2*inter+smooth)/(union+smooth)).mean())
    return sum(losses) / len(losses)


def kl_loss(mu, logvar):
    # Clamp logvar to prevent exp() explosion
    logvar = logvar.clamp(-10, 10)
    mu     = mu.clamp(-10, 10)
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())


# Fix 1: reduce KL weight massively
# Change in total_loss function — find this line:
#   tot += w * (l_ce + l_dc + kl_weight * l_kl)
# Change kl_weight from 5e-5 to 1e-7

def total_loss(outputs, labels, kl_weight=1e-7):  # ← was 5e-5
    ce  = nn.CrossEntropyLoss()
    n   = len(outputs)
    tot = 0.0
    for i, out in enumerate(outputs):
        w     = 0.7 ** (n - 1 - i)
        l_ce  = ce(out['y_logit'], labels)
        l_dc  = multiclass_dice_loss(out['y_logit'], labels)
        l_kl  = kl_loss(out['mu'], out['logvar'])
        tot  += w * (l_ce + l_dc + kl_weight * l_kl)
    return tot / n


def compute_metrics(logits, labels, num_classes=4):
    pred  = logits.argmax(dim=1)
    dices = {}
    names = {1:'rv', 2:'myo', 3:'lv'}
    for c in range(1, num_classes):
        p     = (pred == c).float()
        g     = (labels == c).float()
        inter = (p * g).sum()
        union = p.sum() + g.sum()
        dices[names[c]] = ((2*inter+1e-5)/(union+1e-5)).item()
    dices['mean'] = sum(dices.values()) / len(dices)
    return dices


In [24]:
def run_epoch(model, loader, optimizer=None, train=True):
    model.train() if train else model.eval()
    total_loss_val = 0.0
    all_dices = []
    n = 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            images = batch['image'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            if train:
                optimizer.zero_grad(set_to_none=True)

            outputs = model(images)
            loss    = total_loss(outputs, labels)

            if not torch.isfinite(loss):
                print('  WARNING: inf/nan loss — skipping batch')
                continue

            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            dices = compute_metrics(outputs[-1]['y_logit'], labels)
            all_dices.append(dices)
            total_loss_val += loss.item()
            n += 1

    avg_loss = total_loss_val / max(n, 1)
    avg = lambda k: sum(d[k] for d in all_dices) / max(len(all_dices), 1)
    return {'loss': avg_loss, 'mean': avg('mean'),
            'rv': avg('rv'), 'myo': avg('myo'), 'lv': avg('lv')}


# Run directory
run_dir = Path(REPO_DIR) / 'RESULTS' / 'acdc_v7balanced' / 'run_001'
run_dir.mkdir(parents=True, exist_ok=True)
print('Run dir:', run_dir)

optimizer = torch.optim.AdamW([
    {'params': model.encoder.parameters(),   'lr': LR_ENCODER},
    {'params': model.z_proj.parameters(),    'lr': LR_DECODER},
    {'params': model.low.parameters(),       'lr': LR_DECODER},
    {'params': model.high.parameters(),      'lr': LR_DECODER},
    {'params': model.decoder.parameters(),   'lr': LR_DECODER},
    {'params': model.unc_head.parameters(),  'lr': LR_DECODER},
    {'params': model.halt_head.parameters(), 'lr': LR_DECODER},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_dice = 0.0
history   = []

print(f'\n{"Ep":>4} {"Loss":>8} {"Mean":>8} {"RV":>8} {"Myo":>8} {"LV":>8}')
print('-'*50)

for epoch in range(1, EPOCHS+1):
    t0      = time.time()
    train_m = run_epoch(model, train_loader, optimizer, train=True)
    scheduler.step()
    val_m   = run_epoch(model, val_loader, train=False)
    elapsed = time.time() - t0

    history.append({'epoch': epoch, 'loss': train_m['loss'],
                    'val_mean': val_m['mean'], 'val_rv': val_m['rv'],
                    'val_myo': val_m['myo'], 'val_lv': val_m['lv']})

    if epoch % 10 == 0 or epoch <= 5:
        print(f'{epoch:>4} {train_m["loss"]:>8.4f} '
              f'{val_m["mean"]:>8.4f} {val_m["rv"]:>8.4f} '
              f'{val_m["myo"]:>8.4f} {val_m["lv"]:>8.4f}  t={elapsed:.0f}s')

    if val_m['mean'] > best_dice:
        best_dice = val_m['mean']
        torch.save(model.state_dict(), run_dir / 'best.pth')
        print(f'  -> New best: {best_dice:.4f}')

print(f'\nDone. Best val Dice: {best_dice:.4f}')


Run dir: C:\Users\Debabrata\Desktop\Bright\BRIGHT\SEG_REASONING-main\RESULTS\acdc_v7balanced\run_001

  Ep     Loss     Mean       RV      Myo       LV
--------------------------------------------------
   1   0.5642   0.7473   0.7661   0.6739   0.8019  t=23s
  -> New best: 0.7473
   2   0.2739   0.8200   0.7951   0.7681   0.8968  t=19s
  -> New best: 0.8200
   3   0.2290   0.8506   0.8301   0.7947   0.9269  t=19s
  -> New best: 0.8506
   4   0.2094   0.8563   0.8210   0.8283   0.9195  t=19s
  -> New best: 0.8563
   5   0.1916   0.8431   0.8581   0.7580   0.9130  t=19s
  -> New best: 0.8822
  -> New best: 0.8942
  10   0.1513   0.8866   0.8755   0.8415   0.9429  t=19s
  -> New best: 0.9020
  -> New best: 0.9028
  -> New best: 0.9038
  20   0.1029   0.9015   0.8943   0.8710   0.9394  t=19s
  -> New best: 0.9063
  -> New best: 0.9074
  -> New best: 0.9093
  30   0.0945   0.9058   0.8905   0.8744   0.9527  t=19s
  -> New best: 0.9105
  -> New best: 0.9137
  40   0.0833   0.9146   0.9038  

In [26]:
# Debug NaN loss
model.train()
batch = next(iter(train_loader))
images = batch['image'].to(DEVICE)
labels = batch['label'].to(DEVICE)

print("images:", images.shape, images.min().item(), images.max().item())
print("labels:", labels.shape, labels.unique())

with torch.no_grad():
    outputs = model(images, force_steps=1)
    out = outputs[0]
    print("y_logit:", out['y_logit'].shape, 
          out['y_logit'].min().item(), out['y_logit'].max().item())
    print("y_prob:", out['y_prob'].min().item(), out['y_prob'].max().item())
    print("uncertainty:", out['uncertainty'].min().item(), out['uncertainty'].max().item())
    print("halt_score:", out['halt_score'].min().item(), out['halt_score'].max().item())
    print("mu:", out['mu'].min().item(), out['mu'].max().item())
    print("logvar:", out['logvar'].min().item(), out['logvar'].max().item())

    # Check each loss component
    import torch.nn as nn
    ce = nn.CrossEntropyLoss()
    l_ce   = ce(out['y_logit'], labels)
    l_dice = multiclass_dice_loss(out['y_logit'], labels)
    l_kl   = kl_loss(out['mu'], out['logvar'])
    print()
    print(f"CE loss   : {l_ce.item():.4f}")
    print(f"Dice loss : {l_dice.item():.4f}")
    print(f"KL loss   : {l_kl.item():.4f}")

images: torch.Size([16, 3, 224, 224]) 0.0 1.0
labels: torch.Size([16, 224, 224]) tensor([0, 1, 2, 3], device='cuda:0')
y_logit: torch.Size([16, 4, 224, 224]) -146.54385375976562 83.64970397949219
y_prob: 0.3892251253128052 1.0
uncertainty: 0.2584421634674072 0.8063228726387024
halt_score: 0.17374187707901 1.0
mu: -503.7232666015625 514.3964233398438
logvar: -283.4342346191406 5.193582534790039

CE loss   : 0.0184
Dice loss : 0.0595
KL loss   : 44.3929


In [27]:
model.load_state_dict(torch.load(run_dir / 'best.pth', map_location=DEVICE))
model.eval()
test_m = run_epoch(model, test_loader, train=False)

print('='*55)
print('FINAL TEST — ACDC Cardiac MRI')
print('='*55)
print(f'Mean Dice : {test_m["mean"]:.4f}')
print(f'RV   Dice : {test_m["rv"]:.4f}')
print(f'Myo  Dice : {test_m["myo"]:.4f}')
print(f'LV   Dice : {test_m["lv"]:.4f}')
print()
print('Cross-dataset comparison:')
print(f'  Kvasir-SEG (polyp) : 0.9236 Dice')
print(f'  ACDC (cardiac MRI) : {test_m["mean"]:.4f} Dice')
print('='*55)

FINAL TEST — ACDC Cardiac MRI
Mean Dice : 0.9322
RV   Dice : 0.9333
Myo  Dice : 0.8984
LV   Dice : 0.9650

Cross-dataset comparison:
  Kvasir-SEG (polyp) : 0.9236 Dice
  ACDC (cardiac MRI) : 0.9322 Dice


In [19]:
# Run this to see what we have
test_files = list(Path(DATA_DIR, 'test').glob('*.npz'))
train_files = list(Path(DATA_DIR, 'train').glob('*.npz'))

# Extract unique case IDs
test_cases  = set(f.name.split('_slice')[0] for f in test_files)
train_cases = set(f.name.split('_slice')[0] for f in train_files)

print(f'Train files : {len(train_files)}')
print(f'Test files  : {len(test_files)}')
print(f'Train cases : {len(train_cases)}')
print(f'Test cases  : {len(test_cases)}')
print(f'Test case IDs: {sorted(test_cases)}')

Train files : 1304
Test files  : 40
Train cases : 70
Test cases  : 40
Test case IDs: ['case_002_volume_ED.npz', 'case_002_volume_ES.npz', 'case_003_volume_ED.npz', 'case_003_volume_ES.npz', 'case_008_volume_ED.npz', 'case_008_volume_ES.npz', 'case_009_volume_ED.npz', 'case_009_volume_ES.npz', 'case_012_volume_ED.npz', 'case_012_volume_ES.npz', 'case_014_volume_ED.npz', 'case_014_volume_ES.npz', 'case_017_volume_ED.npz', 'case_017_volume_ES.npz', 'case_024_volume_ED.npz', 'case_024_volume_ES.npz', 'case_042_volume_ED.npz', 'case_042_volume_ES.npz', 'case_048_volume_ED.npz', 'case_048_volume_ES.npz', 'case_049_volume_ED.npz', 'case_049_volume_ES.npz', 'case_053_volume_ED.npz', 'case_053_volume_ES.npz', 'case_055_volume_ED.npz', 'case_055_volume_ES.npz', 'case_064_volume_ED.npz', 'case_064_volume_ES.npz', 'case_067_volume_ED.npz', 'case_067_volume_ES.npz', 'case_079_volume_ED.npz', 'case_079_volume_ES.npz', 'case_081_volume_ED.npz', 'case_081_volume_ES.npz', 'case_088_volume_ED.npz', 'cas